In [1]:
import time
import pandas as pd
from transformers import pipeline

test_df = pd.read_parquet("../data/processed/test_preds.parquet")

clf = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    truncation=True,
    max_length=512,
    batch_size=32,
)

sample = test_df.head(100)
start = time.time()
out = clf(sample["text"].tolist())
print(f"{time.time() - start:.1f} seconds for 100 reviews")

sample = sample.copy()
sample["pred_bert"] = [o["label"].lower() for o in out]
print((sample["pred_bert"] == sample["sentiment"]).mean())
print(out[:3])

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

c:\Users\neelg\Desktop\review-insights\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\neelg\.cache\huggingface\hub\models--distilbert-base-uncased-finetuned-sst-2-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\neelg\Desktop\review-insights\.venv\Lib\site-packages\huggingface_hu

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

8.0 seconds for 100 reviews
0.89
[{'label': 'NEGATIVE', 'score': 0.9851242899894714}, {'label': 'NEGATIVE', 'score': 0.9997015595436096}, {'label': 'NEGATIVE', 'score': 0.9982380867004395}]


In [2]:
import time
import pandas as pd
from transformers import pipeline

test_df = pd.read_parquet("../data/processed/test_preds.parquet")

clf = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    truncation=True,
    max_length=512,
    batch_size=32,
)

start = time.time()
out = clf(test_df["text"].tolist())
print(f"{(time.time() - start) / 60:.1f} minutes")

test_df["pred_bert"] = [o["label"].lower() for o in out]
test_df["bert_conf"] = [o["score"] for o in out]
test_df.to_parquet("../data/processed/test_preds.parquet", index=False)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

31.7 minutes


In [3]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

y = test_df["sentiment"]
p = test_df["pred_bert"]

print("Accuracy:", accuracy_score(y, p))
print("F1 (macro):", f1_score(y, p, average="macro"))
print(classification_report(y, p))
print(confusion_matrix(y, p, labels=["negative", "positive"]))

Accuracy: 0.898
F1 (macro): 0.8979989555093044
              precision    recall  f1-score   support

    negative       0.88      0.91      0.90      4934
    positive       0.91      0.88      0.90      5066

    accuracy                           0.90     10000
   macro avg       0.90      0.90      0.90     10000
weighted avg       0.90      0.90      0.90     10000

[[4506  428]
 [ 592 4474]]


In [4]:
tfidf_ok = test_df["pred_tfidf"] == y
bert_ok = test_df["pred_bert"] == y

print("Both right:      ", (tfidf_ok & bert_ok).sum())
print("Only TF-IDF right:", (tfidf_ok & ~bert_ok).sum())
print("Only BERT right:  ", (~tfidf_ok & bert_ok).sum())
print("Both wrong:       ", (~tfidf_ok & ~bert_ok).sum())

Both right:       8424
Only TF-IDF right: 593
Only BERT right:   556
Both wrong:        427


In [5]:
y = test_df["sentiment"]
only_tfidf = test_df[(test_df["pred_tfidf"] == y) & (test_df["pred_bert"] != y)]
only_bert = test_df[(test_df["pred_tfidf"] != y) & (test_df["pred_bert"] == y)]
both_wrong = test_df[(test_df["pred_tfidf"] != y) & (test_df["pred_bert"] != y)]

for name, d in [("ONLY TF-IDF RIGHT", only_tfidf),
                ("ONLY BERT RIGHT", only_bert),
                ("BOTH WRONG", both_wrong)]:
    print("=====", name)
    for _, r in d.sample(8, random_state=1).iterrows():
        print(f"TRUE: {r['sentiment']} | BERT conf: {r['bert_conf']:.2f}")
        print(r["text"][:250], "\n")

===== ONLY TF-IDF RIGHT
TRUE: negative | BERT conf: 0.74
This book is treason: I read this book back in the sixties. It was a sick anti-American tirade then and it has not gotten any better with age. Historians will find this book fascinating for documenting the depths to which America can fall. I am appal 

TRUE: positive | BERT conf: 0.73
WII Remote: Great....really like to rubber cover for better grip and protection if it flies out your hand. 

TRUE: positive | BERT conf: 0.99
A Must Have for First Time Parents!: During my daughter's third week, she began to cry consistently for several hours in the evening. My husband and I were convinced it was colic. My sister-in-law, knowing how exhausted we were, researched informatio 

TRUE: negative | BERT conf: 0.99
Manhattan Toy Stella Doll: I decided not to give to 18 month old granddaughter because there is a magnet in the pacifer and a magnet under one layer of cloth on the doll and was afraid she would bite on it and somehow get to the 